# 03. Làm sạch và tạo đặc trưng dữ liệu

Notebook này loại bỏ các dòng thiếu ngày commit, gộp các khối code trùng lặp, tạo thêm các cột đặc trưng mới (vai trò file, mục đích sửa đổi, cửa sổ 30 ngày) và xuất ra dữ liệu sạch để phân tích.

Import thư viện và tải dữ liệu chuẩn hóa.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
DATA_PATH = Path("../data/interim/latest_standardized_data.csv")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {DATA_PATH}: {len(df):,} rows x {df.shape[1]} columns")

Loaded ../data/interim/latest_standardized_data.csv: 35,278 rows x 12 columns


Chuyển các cột ngày tháng về định dạng UTC và tạo cột `has_recorded_change` đánh dấu những khối có ghi nhận lần thay đổi đầu tiên.

In [2]:
for col in ["commit_date", "change_commit_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df["has_recorded_change"] = df["change_commit_date"].notna()
df[["commit_date", "change_commit_date", "change_commit_url"]].isna().sum()

commit_date            1642
change_commit_date    22282
change_commit_url     22282
dtype: int64

Loại bỏ các dòng thiếu `commit_date`, gộp các dòng trùng theo cặp `(repo, file, code_block)` và bỏ dữ liệu tháng 4/2026 do chưa đủ tháng.

In [3]:
KEY = ["repo", "file", "code_block"]
n_raw = len(df)

df_clean = df.dropna(subset=["commit_date"]).copy()
n_dated = len(df_clean)

df_clean = (
    df_clean.sort_values(["change_commit_date", "commit_date"], na_position="last")
    .drop_duplicates(KEY)
    .sort_index()
)
n_unique = len(df_clean)

df_clean = df_clean[df_clean["commit_date"] < pd.Timestamp("2026-04-01", tz="UTC")].copy()

print(f"Dropped {n_raw - n_dated:,} rows missing commit_date")
print(f"Merged {n_dated - n_unique:,} duplicate rows into {n_unique:,} unique blocks")
print(f"Dropped {n_unique - len(df_clean):,} April 2026 blocks -> {len(df_clean):,} blocks remaining")

Dropped 1,642 rows missing commit_date
Merged 12,345 duplicate rows into 21,291 unique blocks
Dropped 4 April 2026 blocks -> 21,287 blocks remaining


Tạo các cột thời gian và tính số ngày đến lần sửa đầu tiên (`days_to_modification`). Các trường hợp nhỏ hơn hoặc bằng 0 (do dòng comment bị sửa sau khi code đã đổi) được chuyển thành giá trị trống.

In [4]:
df_clean["year"] = df_clean["commit_date"].dt.year.astype("Int64")
df_clean["month"] = df_clean["commit_date"].dt.month.astype("Int64")
df_clean["year_month"] = df_clean["commit_date"].dt.strftime("%Y-%m")
df_clean["quarter"] = "Q" + df_clean["commit_date"].dt.quarter.astype(str)
df_clean["day_of_week"] = df_clean["commit_date"].dt.day_name()
df_clean["is_weekend"] = df_clean["commit_date"].dt.dayofweek >= 5

df_clean["days_to_modification"] = (
    (df_clean["change_commit_date"] - df_clean["commit_date"]).dt.total_seconds() / 86400
).round(3)

invalid_days = int((df_clean["days_to_modification"] <= 0).sum())
df_clean.loc[df_clean["days_to_modification"] <= 0, "days_to_modification"] = np.nan
print(f"Converted {invalid_days:,} non-positive modification intervals to NaN")

Converted 1,731 non-positive modification intervals to NaN


Tạo biến `change_30d` để kiểm tra khối code có bị sửa trong vòng 30 ngày đầu hay không, chỉ tính trên những khối có đủ ít nhất 30 ngày theo dõi để so sánh công bằng giữa các năm.

In [5]:
observation_end = df_clean["change_commit_date"].max()
has_30d_window = df_clean["commit_date"] <= (observation_end - pd.Timedelta(days=30))
valid_timing = ~(df_clean["has_recorded_change"] & df_clean["days_to_modification"].isna())

df_clean["change_30d"] = pd.Series(pd.NA, index=df_clean.index, dtype="boolean")
eligible = has_30d_window & valid_timing
df_clean.loc[eligible, "change_30d"] = df_clean.loc[eligible, "days_to_modification"].le(30).fillna(False)

print(f"Eligible for 30-day window: {eligible.sum():,} blocks; 30-day change rate: {df_clean.loc[eligible, 'change_30d'].mean():.1%}")

Eligible for 30-day window: 19,554 blocks; 30-day change rate: 23.0%


Tính độ dài của đoạn code, độ dài comment và kiểm tra xem đoạn code có chứa định nghĩa hàm hoặc class hay không.

In [7]:
code = df_clean["code_block"].fillna("").astype(str)
newline_pat = "\n" if code.str.contains("\n", regex=False).any() else r"\\n"
df_clean["code_lines"] = code.str.count(newline_pat).add(code.ne("").astype(int))
df_clean["code_chars"] = code.str.len()
df_clean["has_function_def"] = code.str.contains(r"\bdef\s+\w+|\bfunction\s+\w+|=>", regex=True)
df_clean["has_class_def"] = code.str.contains(r"\bclass\s+\w+", regex=True)

if "comment" in df_clean.columns:
    comm = df_clean["comment"].fillna("").astype(str)
else:
    comm = df_clean["id"].str.split("::", n=3).str[3].fillna("").astype(str)

df_clean["comment_chars"] = comm.str.len()

Nhận diện công cụ AI (`ai_tool`), mức độ hỗ trợ (`ai_verb`) từ nội dung comment và đếm số commit AI của mỗi repo trong tập dữ liệu (`repo_ai_commit_count`).

In [8]:
def first_label(series, rules, default):
    result = pd.Series(default, index=series.index, dtype="object")
    for label, pat in rules:
        result.loc[result.eq(default) & series.str.contains(pat, regex=True, na=False)] = label
    return result

pattern = df_clean["id"].str.split("__").str[1].str.replace("_", " ")
tool_text = (pattern + " " + comm).str.lower()

df_clean["ai_tool"] = first_label(
    tool_text,
    [
        ("chatgpt_gpt", r"chatgpt|gpt|openai"),
        ("github_copilot", r"copilot"),
        ("claude", r"claude|anthropic"),
        ("gemini_bard", r"gemini|bard"),
        ("llama_meta", r"llama|meta ai"),
        ("other_tool", r"deepseek|codewhisperer|tabnine|codeium|cursor (?:ai|ide|editor|agent)|(?:by|with|using|in|via) cursor\b"),
    ],
    default="other_ai",
)
df_clean["ai_verb"] = np.where(pattern.str.contains("assist|suggest"), "assisted", "written")
df_clean["repo_ai_commit_count"] = df_clean.groupby("repo")["commit_url"].transform("nunique")

Phân loại vai trò của file (`file_role`) từ đường dẫn và phân loại mục đích của lần sửa đầu tiên (`change_intent`) từ commit message.

In [9]:
file_lower = df_clean["file"].fillna("").str.lower()
df_clean["file_role"] = first_label(
    file_lower,
    [
        ("test", r"(?:^|/)(?:tests?|specs?|__tests__)/|_test\.|\.test\.|\.spec\.|test_"),
        ("notebook_or_script", r"\.ipynb$|aoc|advent|puzzle|script|playground|example|demo|tutorial"),
        ("docs_or_config", r"docs?/|readme|config|setup\.py|settings"),
        ("frontend_web", r"components?/|pages?/|views?/|public/|static/|assets?/|styles?/|\.jsx$|\.mjs$"),
    ],
    default="core_source",
)

if "change_commit_message" in df_clean.columns:
    msg_lower = df_clean["change_commit_message"].fillna("").str.lower()
    df_clean["change_intent"] = first_label(
        msg_lower,
        [
            ("bug_fix", r"\bfix(?:ed|es|ing)?\b|\bbug\b|\berror\b|\bissue\b|\bpatch\b|\bhotfix\b|\bcrash\b"),
            ("refactor_cleanup", r"\brefactor|\bclean|\blint|\bformat|\brename|\btypo|\bsimplif|\bstyle\b"),
            ("docs_or_test", r"\btest|\bdoc(?:s|ument)?\b|\breadme\b|\bcomment"),
            ("feature_update", r"\badd(?:ed|s|ing)?\b|\bfeat|\bupdate|\bimplement|\bsupport|\bimprove|\benhance"),
        ],
        default="other_change",
    )
    df_clean.loc[~df_clean["has_recorded_change"], "change_intent"] = "no_recorded_change"
else:
    df_clean["change_intent"] = np.where(df_clean["has_recorded_change"], "other_change", "no_recorded_change")

Xem bảng thống kê mô tả các biến số và tần suất xuất hiện của các nhóm phân loại chính.

In [10]:
numeric_cols = ["days_to_modification", "code_lines", "code_chars", "comment_chars", "repo_ai_commit_count"]
description = df_clean[numeric_cols].describe().T
description["skew"] = df_clean[numeric_cols].skew()
display(description)

for col in ["language", "ai_tool", "file_role", "change_intent", "has_recorded_change"]:
    print(f"\nFrequency: {col}")
    display(df_clean[col].value_counts(dropna=False).to_frame("count"))

,count,mean,std,min,25%,50%,75%,max,skew
days_to_modification,5714.0,29.112097,71.502392,0.001,0.27625,3.067,21.03275,1011.911,5.175990
code_lines,21287.0,96.140837,186.959517,2.000,12.00000,32.000,88.00000,2833.000,4.761274
code_chars,21287.0,3623.104383,7353.016068,1.000,387.00000,1114.000,3202.50000,98973.000,4.765625
comment_chars,21287.0,356.729929,1470.221783,12.000,51.00000,95.000,274.00000,115215.000,32.292749
repo_ai_commit_count,21287.0,13.024334,37.464013,1.000,1.00000,1.000,2.00000,141.000,3.101625



Frequency: language


,count
language,
python,15624
js,5663



Frequency: ai_tool


,count
ai_tool,
other_ai,6507
chatgpt_gpt,6408
claude,4160
github_copilot,2691
gemini_bard,1101
llama_meta,387
other_tool,33



Frequency: file_role


,count
file_role,
core_source,15708
notebook_or_script,2011
test,1533
frontend_web,1513
docs_or_config,522



Frequency: change_intent


,count
change_intent,
no_recorded_change,13842
feature_update,2132
other_change,2054
bug_fix,1905
refactor_cleanup,752
docs_or_test,602



Frequency: has_recorded_change


,count
has_recorded_change,
False,13842
True,7445


Kiểm tra các điều kiện chất lượng dữ liệu và xuất bảng kết quả cuối cùng ra file `data/processed/commits_features.csv`.

In [11]:
assert df_clean["commit_date"].notna().all()
assert not df_clean.duplicated(KEY).any()
assert df_clean["has_recorded_change"].dtype == bool
assert (df_clean["days_to_modification"].dropna() > 0).all()

output_path = Path("../data/processed/commits_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)
print(f"Exported {output_path} ({df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns)")

Exported ../data/processed/commits_features.csv (21,287 rows x 31 columns)
